# Week 1-2

In [ ]:
import math

# ── CHANGE ONLY THIS ──────────────────────
RIM_RADIUS   = 0.3   # metres  ← your one parameter
# ─────────────────────────────────────────

N            = 16
CENTER       = (6.1, 0, 3)
TUBE_RADIUS  = 0.02
RGBA         = "1.0 0.4 0.0 1"
CONAFFINITY  = "1"
CONTYPE      = "0"

seg_half = math.pi * RIM_RADIUS / N
cx, cy, cz = CENTER

print(f"<!-- Rim: R={RIM_RADIUS}, N={N}, tube={TUBE_RADIUS} -->")
for i in range(N):
    theta = 2 * math.pi * i / N
    px = cx + RIM_RADIUS * math.cos(theta)
    py = cy + RIM_RADIUS * math.sin(theta)
    ax = -math.cos(theta)
    ay = -math.sin(theta)
    print(
        f'<geom type="cylinder" '
        f'size="{TUBE_RADIUS} {seg_half:.5f}" '
        f'pos="{px:.4f} {py:.4f} {cz}" '
        f'axisangle="{ax:.4f} {ay:.4f} 0 1.5708" '
        f'conaffinity="{CONAFFINITY}" contype="{CONTYPE}" '
        f'rgba="{RGBA}"/>'
    )

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Reacher-v5", xml_file =r"E:\personal projects\FAFO-RL\models\reacher.xml", render_mode = "human")
observation, info = env.reset(seed=42)


print("Obs shape:", observation.shape)
print("Action space:", env.action_space)

try:
    for i in range(1000):  # just 5 steps to see info
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i} | reward: {reward:.4f} | info: {info}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Pendulum-v1")
model = SAC("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=20000)
print("Stack working")

In [ ]:
env = gym.make("Pendulum-v1", render_mode = "human")


obs, info = env.reset(seed=42)

try:
    total_reward = 0
    for i in range(1000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
            print(total_reward)
            observation, info = env.reset()
            total_reward = 0
finally:
    env.close()

## Reacher SAC

In [ ]:
import numpy as np
import gymnasium as gym
class CustomPendulumEnv(gym.Wrapper):
    def angle_normalize(self, x):
        return ((x + np.pi) % (2 * np.pi)) - np.pi

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # obs = [cos(th), sin(th), thdot]
        th = np.arctan2(obs[1], obs[0])   # recover angle from cos/sin
        thdot = obs[2]
        u = np.clip(action, -2.0, 2.0)[0]

        # your custom reward
        costs = self.angle_normalize(th)**2 + 0.1 * thdot**2
        modified_reward = -costs           # dropped action cost term

        return obs, modified_reward, terminated, truncated, info

In [ ]:
import inspect
# import gymnasium as gym

# Create the base environment (unwrapped to bypass time limits/wrappers)
env = CustomPendulumEnv(gym.make("Pendulum-v1"))

# Print the source code of the step function where the reward is calculated
print(inspect.getsource(env.step))

In [ ]:
from stable_baselines3 import SAC, PPO
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=20e3, tb_log_name="pendulum_modifield_reward")
print("Stack working")

## Different Seed Testing

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO
env = gym.make("Reacher-v5")
# get_device('cpu')
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=500e3, tb_log_name="reacher_sac_long")
# model = SAC("MlpPolicy", env, verbose=1,seed = 33, device="cpu", tensorboard_log = r".\log")
# model.learn(total_timesteps=20e3, tb_log_name="pendulum_different_seed")
print("Stack working")

In [ ]:
model.save(r".\models\reacher_sac_500k")

In [ ]:
del model

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import SAC

# class MountainCarWrapper(gym.Wrapper):
#     def step(self, action):
#         obs, reward, terminated, truncated, info = self.env.step(action)
#         position, velocity = obs
#         reward += 10 * abs(velocity)
#         reward += 3 * (position + 0.5)
#         return obs, reward, terminated, truncated, info

# env = MountainCarWrapper(gym.make("MountainCarContinuous-v0"))
env = gym.make("MountainCarContinuous-v0")
model = SAC("MlpPolicy", env,
            learning_starts=1000,
            verbose=1,
            tensorboard_log=r".\log",
            seed=42)
model.learn(total_timesteps=100_000,
            tb_log_name="SAC_MountainCar_normal")

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO

model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=100e3, tb_log_name="mountain_car_continuous_reward_wrapper")
print("Stack working")

In [ ]:
# import gymnasium as gym
# from stable_baselines3 import SAC, PPO

env = gym.make("MountainCarContinuous-v0", render_mode = "human")
# model = SAC.load(r".\models\reacher_sac_500k", env=env)

# obs, info = env.reset(seed=32)
obs, info = env.reset()

try:
    total_reward = 0
    for i in range(10000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
        # print(total_reward)
            observation, info = env.reset()
        total_reward = 0
finally:
    env.close()

# Week 3

In [ ]:
import mujoco

model = mujoco.MjModel.from_xml_path(
    r"E:\personal projects\FAFO-RL\agnet_xml\three_pointer.xml"
)
data = mujoco.MjData(model)
print("Bodies  :", model.nbody)
print("Joints  :", model.njnt)
print("Actuators:", model.nu)
print("qpos shape:", data.qpos.shape)
print("Valid ✓")

In [8]:
# import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC
from pathlib import Path
# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)
current_dir = Path.cwd()
xml_path_str = str(current_dir.parent / "agnet_xml" / "three_pointer.xml")
env = gym.make(
    "ThreePointer-v0",
    xml_file=xml_path_str,
    render_mode="human",
)

observation, info = env.reset(seed=42)
print("Obs shape:", observation.shape)
print("Action space:", env.action_space)

try:
    for i in range(10000):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"{i}", f"Observation {observation[7]}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()

c:\vsr\Naren\FAFO-RL\rll_venv\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment ThreePointer-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


Obs shape: (10,)
Action space: Box(-1.0, 1.0, (2,), float32)
0 Observation 0.0
1 Observation 0.0
2 Observation 0.0
3 Observation 0.0
4 Observation 0.0
5 Observation 0.0
6 Observation 0.0
7 Observation 0.0
8 Observation 0.0
9 Observation 0.0
10 Observation 0.0
11 Observation 0.0
12 Observation 0.0
13 Observation 0.0
14 Observation 0.0
15 Observation 0.0
16 Observation 0.0
17 Observation 0.0
18 Observation 0.0
19 Observation 0.0
20 Observation 0.0
21 Observation 0.0
22 Observation 0.0
23 Observation 0.0
24 Observation 0.0
25 Observation 0.0
26 Observation 0.0
27 Observation 0.0
28 Observation 0.0
29 Observation 0.0
30 Observation 0.0
31 Observation 0.0
32 Observation 0.0
33 Observation 0.0
34 Observation 0.0
35 Observation 0.0
36 Observation 0.0
37 Observation 0.0
38 Observation 0.0
39 Observation 0.0
40 Observation 0.0
41 Observation 0.0
42 Observation 0.0
43 Observation 0.0
44 Observation 0.0
45 Observation 0.0
46 Observation 0.0
47 Observation 0.0
48 Observation 0.0
49 Observation 0.0

In [ ]:
# import mujoco
from stable_baselines3 import SAC, PPO

model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\three_pointer_log")
model.learn(total_timesteps=100e3, tb_log_name="default_settings")
print("Stack working")

In [ ]:
model.save(r".\models\three_pointer_sac_100k")


In [ ]:
import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    xml_file=r"E:\personal projects\FAFO-RL\agnet_xml\three_pointer.xml",
    render_mode="human",
)
model = SAC.load(r"E:\personal projects\FAFO-RL\scripts\models\three_pointer_sac_100k.zip", env=env)
observation, info = env.reset()          # unpack the tuple — this was the crash

for i in range(10000):
    action, _states = model.predict(observation, deterministic=True)
    observation, reward, terminated, truncated, info = env.step(action)
    print(f"{i}", f"Observation {observation[7]}")
    if terminated or truncated:
        observation, info = env.reset()